# 02 — Data Cleaning & Chunking
## Cleaning 30,444 legal sections and splitting into chunks
We will:
- Delete 450 empty Content rows
- Clean messy text
- Split long sections into 500 character chunks
- Save as processed_chunks.json



In [1]:
import pandas as pd
import json
import re
import os

# Load the Excel file again
df = pd.read_excel('../data/Scrapmetadata_forvectordatabase.xlsx')

print("✅ File loaded!")
print(f"📊 Total rows before cleaning: {len(df)}")

# What is this cell doing?
# Every notebook starts fresh — it does not remember what previous notebooks did. So we load the Excel file again here.
# Think of it like this — every notebook is a new classroom. When you enter a new classroom, you bring your textbook again. You don't expect the previous classroom to pass it for you!

✅ File loaded!
📊 Total rows before cleaning: 30444


## Step 1 — Remove Empty Content Rows
Delete all rows where Content is empty/missing

In [ ]:
# Remove rows where Content is empty
df_clean = df.dropna(subset=['Content'])

print(f"📊 Rows before cleaning : {len(df)}")
print(f"📊 Rows after cleaning  : {len(df_clean)}")
print(f"🗑️  Rows deleted         : {len(df) - len(df_clean)}")
print()

# Also remove rows where Content is too short (less than 10 characters)
df_clean = df_clean[df_clean['Content'].str.len() > 10]

print(f"📊 Rows after removing very short sections: {len(df_clean)}")
print()
print("✅ Empty rows removed successfully!")

# Note - : 🎉 450 empty rows deleted! Data is cleaner now!

📊 Rows before cleaning : 30444
📊 Rows after cleaning  : 29994
🗑️  Rows deleted         : 450

📊 Rows after removing very short sections: 29992

✅ Empty rows removed successfully!


## Step 2 — Clean Messy Text
Remove extra spaces, special characters and fix formatting

In [3]:
# Clean the text content
def clean_text(text):
    text = str(text)                      # make sure it is a string
    text = text.strip()                   # remove spaces from start and end
    text = re.sub(r'\s+', ' ', text)      # remove extra spaces between words
    text = re.sub(r'\n+', ' ', text)      # remove new lines
    text = re.sub(r'\t+', ' ', text)      # remove tabs
    return text

# Apply cleaning to Content column
df_clean['Content'] = df_clean['Content'].apply(clean_text)

# Also clean Section Heading
df_clean['Section Heading'] = df_clean['Section Heading'].apply(clean_text)

# Also clean Act Title
df_clean['Act Title'] = df_clean['Act Title'].apply(clean_text)

print("✅ Text cleaning done!")
print()
print("📖 Sample cleaned content:")
print("-" * 60)
print(df_clean['Content'].iloc[0][:300])

✅ Text cleaning done!

📖 Sample cleaned content:
------------------------------------------------------------
(1) This Act may be called the Building and Other Construction Workers’ Welfare Cess Act, 1996. (2) It extends to the whole of India. (3) It shall be deemed to have come into force on the 3rd day of November, 1995.


## Step 3 — Chunking
Split long sections into 500 character pieces
This is critical for FAISS to work properly

In [4]:
# Chunking function
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap  # overlap keeps context between chunks
    return chunks

# Test chunking on one section
sample_text = df_clean['Content'].iloc[5]
print(f"📏 Original length : {len(sample_text)} characters")
print()
chunks = chunk_text(sample_text)
print(f"✂️  Number of chunks: {len(chunks)}")
print()
print("📖 First chunk:")
print("-" * 60)
print(chunks[0])
print()
print("📖 Second chunk (notice 50 char overlap with first):")
print("-" * 60)
print(chunks[1] if len(chunks) > 1 else "Only one chunk!")

📏 Original length : 315 characters

✂️  Number of chunks: 1

📖 First chunk:
------------------------------------------------------------
Notwithstanding anything contained in this Act, the Central Government may, by notification in the Official Gazette, exempt any employer or class of employers in a State from the payment of cess payable under this Act where such cess is already levied and payable under any corresponding law in force in that State.

📖 Second chunk (notice 50 char overlap with first):
------------------------------------------------------------
Only one chunk!


## Step 4 — Apply Chunking to All Sections
Split ALL 29,000+ sections into chunks

In [5]:
# Apply chunking to ALL sections
all_chunks = []

for _, row in df_clean.iterrows():
    chunks = chunk_text(row['Content'])
    
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'text': chunk,
            'act_title': row['Act Title'],
            'act_id': row['Act ID'],
            'section_id': row['Section ID'],
            'section_heading': row['Section Heading'],
            'chapter_name': str(row['Chapter Name']),
            'enactment_date': str(row['Enactment Date']),
            'chunk_number': i + 1
        })

print(f"✅ Chunking complete!")
print()
print(f"📊 Original sections : {len(df_clean)}")
print(f"📊 Total chunks made : {len(all_chunks)}")
print(f"📈 Average chunks    : {round(len(all_chunks)/len(df_clean), 2)} per section")
print()
print("📖 Sample chunk with metadata:")
print("-" * 60)
sample = all_chunks[100]
print(f"Act Title   : {sample['act_title']}")
print(f"Section ID  : {sample['section_id']}")
print(f"Chunk No    : {sample['chunk_number']}")
print(f"Text        : {sample['text'][:200]}")

✅ Chunking complete!

📊 Original sections : 29992
📊 Total chunks made : 56617
📈 Average chunks    : 1.89 per section

📖 Sample chunk with metadata:
------------------------------------------------------------
Act Title   : THE INCOME-TAX ACT, 1961
Section ID  : Section 12.
Chunk No    : 2
Text        : s of that section and section 13 shall apply accordingly.] 6[(2) The value of any services, being medical or educational services, made available by any charitable or religious trust running a hospita


## Step 5 — Save Chunks as JSON
Save all 56,617 chunks to disk for FAISS notebook

In [6]:
# Save all chunks to JSON file
output_path = '../data/processed_chunks.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print("✅ Chunks saved successfully!")
print()
print(f"📁 Saved to     : {output_path}")
print(f"📊 Total chunks : {len(all_chunks)}")
print()

# Verify file was saved correctly
with open(output_path, 'r', encoding='utf-8') as f:
    verify = json.load(f)

print(f"✅ Verification  : {len(verify)} chunks loaded back correctly!")
print()
print("📖 First chunk in file:")
print("-" * 60)
print(f"Act Title : {verify[0]['act_title']}")
print(f"Section   : {verify[0]['section_id']}")
print(f"Text      : {verify[0]['text'][:200]}")

✅ Chunks saved successfully!

📁 Saved to     : ../data/processed_chunks.json
📊 Total chunks : 56617

✅ Verification  : 56617 chunks loaded back correctly!

📖 First chunk in file:
------------------------------------------------------------
Act Title : THE BUILDING AND OTHER CONSTRUCTION WORKERS’ WELFARE CESS ACT, 1996
Section   : Section 1.
Text      : (1) This Act may be called the Building and Other Construction Workers’ Welfare Cess Act, 1996. (2) It extends to the whole of India. (3) It shall be deemed to have come into force on the 3rd day of N


## Final Summary — Data Cleaning Complete

In [7]:
print("=" * 60)
print("📊 DATA CLEANING SUMMARY")
print("=" * 60)
print()
print("Step 1 — Removed Empty Rows:")
print(f"   Before : 30,444 sections")
print(f"   After  : 29,992 sections")
print(f"   Deleted: 452 empty/short rows")
print()
print("Step 2 — Text Cleaning:")
print(f"   ✅ Removed extra spaces")
print(f"   ✅ Removed new lines")
print(f"   ✅ Removed tabs")
print()
print("Step 3 — Chunking:")
print(f"   Chunk size : 500 characters")
print(f"   Overlap    : 50 characters")
print(f"   Before     : 29,992 sections")
print(f"   After      : 56,617 chunks")
print(f"   Average    : 1.89 chunks per section")
print()
print("Step 4 — Saved:")
print(f"   📁 ../data/processed_chunks.json")
print(f"   56,617 chunks with metadata")
print()
print("=" * 60)
print("✅ Ready to move to 03_embeddings_faiss.ipynb!")
print("=" * 60)

📊 DATA CLEANING SUMMARY

Step 1 — Removed Empty Rows:
   Before : 30,444 sections
   After  : 29,992 sections
   Deleted: 452 empty/short rows

Step 2 — Text Cleaning:
   ✅ Removed extra spaces
   ✅ Removed new lines
   ✅ Removed tabs

Step 3 — Chunking:
   Chunk size : 500 characters
   Overlap    : 50 characters
   Before     : 29,992 sections
   After      : 56,617 chunks
   Average    : 1.89 chunks per section

Step 4 — Saved:
   📁 ../data/processed_chunks.json
   56,617 chunks with metadata

✅ Ready to move to 03_embeddings_faiss.ipynb!
